# Task 1: Business Questions & Data Cleaning
## HMS - Hotel Booking Demand

## Business Questions
**Q1:** Which bookings will be cancelled, and should we ask those guests for a deposit?  
**Q2:** What is the expected price per night for each booking?  
**Q3:** When is the hotel fully booked, and what drives high-demand periods?

## Imports & Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv('../data/hotel_bookings.csv')

print(f"✅ Data loaded!")
print(f"📊 Shape: {df.shape}")
print(f"📋 Columns: {df.columns.tolist()}")

✅ Data loaded!
📊 Shape: (119390, 32)
📋 Columns: ['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']


In [2]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## Data Types & Missing Values

In [3]:
report = []
for col in df.columns:
    report.append({
        'Column'    : col,
        'Type'      : str(df[col].dtype),
        'Missing'   : df[col].isnull().sum(),
        'Missing_%' : round(df[col].isnull().mean() * 100, 2),
        'Unique'    : df[col].nunique(),
        'Min'       : df[col].min() if df[col].dtype != 'object' else '-',
        'Max'       : df[col].max() if df[col].dtype != 'object' else '-'
    })

pd.DataFrame(report)

,Column,Type,Missing,Missing_%,Unique,Min,Max
0,hotel,object,0,0.00,2,-,-
1,is_canceled,int64,0,0.00,2,0,1
2,lead_time,int64,0,0.00,479,0,737
3,arrival_date_year,int64,0,0.00,3,2015,2017
4,arrival_date_month,object,0,0.00,12,-,-
5,arrival_date_week_number,int64,0,0.00,53,1,53
6,arrival_date_day_of_month,int64,0,0.00,31,1,31
7,stays_in_weekend_nights,int64,0,0.00,17,0,19
8,stays_in_week_nights,int64,0,0.00,35,0,50
9,adults,int64,0,0.00,14,0,55


## Check Problems

In [4]:
print("=" * 50)
print("🔍 CHECKING FOR PROBLEMS")
print("=" * 50)

# 1. Duplicate rows
dups = df.duplicated().sum()
print(f"\n1️⃣ Duplicate rows: {dups}")

# 2. Bookings with no guests
no_guests = df[(df['adults']==0) & (df['children']==0) & (df['babies']==0)]
print(f"2️⃣ Bookings with no guests: {len(no_guests)}")

# 3. Bookings with zero nights
zero_nights = df[(df['stays_in_weekend_nights']==0) & (df['stays_in_week_nights']==0)]
print(f"3️⃣ Bookings with zero nights: {len(zero_nights)}")

# 4. Negative ADR
bad_adr = df[df['adr'] < 0]
print(f"4️⃣ Negative ADR: {len(bad_adr)}")

# 5. Zero ADR
zero_adr = df[df['adr'] == 0]
print(f"5️⃣ Zero ADR: {len(zero_adr)}")

# 6. ADR outliers (> 4000)
high_adr = df[df['adr'] > 4000]
print(f"6️⃣ Extreme ADR (>4000): {len(high_adr)}")

# 7. Adults outliers
high_adults = df[df['adults'] > 10]
print(f"7️⃣ Adults > 10: {len(high_adults)}")
print(f"   Max adults: {df['adults'].max()}")

# 8. is_canceled vs reservation_status consistency
inconsistent = df[
    ((df['is_canceled'] == 1) & (df['reservation_status'] != 'Canceled')) |
    ((df['is_canceled'] == 0) & (df['reservation_status'] == 'Canceled'))
]
print(f"8️⃣ is_canceled vs reservation_status mismatch: {len(inconsistent)}")

# 9. meal values
print(f"\n9️⃣ Meal values:\n{df['meal'].value_counts()}")

# 10. Missing values
print(f"\n🔟 Missing values:")
missing = df.isnull().sum()
print(missing[missing > 0])

# 11. Date column type
print(f"\n1️⃣1️⃣ reservation_status_date type: {df['reservation_status_date'].dtype}")

# 12. arrival_date_month type
print(f"1️⃣2️⃣ arrival_date_month type: {df['arrival_date_month'].dtype}")
print(f"    Values: {df['arrival_date_month'].unique()}")

🔍 CHECKING FOR PROBLEMS

1️⃣ Duplicate rows: 31994
2️⃣ Bookings with no guests: 180
3️⃣ Bookings with zero nights: 715
4️⃣ Negative ADR: 1
5️⃣ Zero ADR: 1959
6️⃣ Extreme ADR (>4000): 1
7️⃣ Adults > 10: 12
   Max adults: 55
8️⃣ is_canceled vs reservation_status mismatch: 1207

9️⃣ Meal values:
meal
BB           92310
HB           14463
SC           10650
Undefined     1169
FB             798
Name: count, dtype: int64

🔟 Missing values:
children         4
country        488
agent        16340
company     112593
dtype: int64

1️⃣1️⃣ reservation_status_date type: object
1️⃣2️⃣ arrival_date_month type: object
    Values: ['July' 'August' 'September' 'October' 'November' 'December' 'January'
 'February' 'March' 'April' 'May' 'June']


## What the Data Doesn't Tell Us:
- We don't know **how** the data was collected (manual? automated?)
- Covers only **2 hotels in Portugal** → cannot generalize to all hotels
- Covers **July 2015 to August 2017** only → no recent trends
- **Personal guest info** is hidden (no names, no emails)
- **agent** and **company** columns use codes → real names unknown
- We cannot tell if **cancelled bookings** were re-booked later
- **No revenue data** beyond ADR (no taxes, extras, F&B)
- **No show** guests are mixed with cancellations in some analyses

## Clean the Data

In [5]:
df_clean = df.copy()

# ── 1. Drop duplicates ───────────────────────────────
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
dropped_dups = before - len(df_clean)
print(f"1️⃣ Dropped duplicates: {dropped_dups} rows")

# ── 2. Drop bookings with no guests ──────────────────
before = len(df_clean)
df_clean = df_clean[~(
    (df_clean['adults']==0) & 
    (df_clean['children']==0) & 
    (df_clean['babies']==0)
)]
dropped_no_guests = before - len(df_clean)
print(f"2️⃣ Dropped no-guest bookings: {dropped_no_guests} rows")

# ── 3. Drop bookings with zero nights ────────────────
before = len(df_clean)
df_clean = df_clean[~(
    (df_clean['stays_in_weekend_nights']==0) & 
    (df_clean['stays_in_week_nights']==0)
)]
dropped_zero_nights = before - len(df_clean)
print(f"3️⃣ Dropped zero-night bookings: {dropped_zero_nights} rows")

# ── 4. Drop negative ADR ─────────────────────────────
before = len(df_clean)
df_clean = df_clean[df_clean['adr'] >= 0]
dropped_bad_adr = before - len(df_clean)
print(f"4️⃣ Dropped negative ADR: {dropped_bad_adr} rows")

# ── 5. Flag extreme ADR (> 4000) ─────────────────────
df_clean['adr_extreme_flag'] = (df_clean['adr'] > 4000).astype(int)
print(f"5️⃣ Flagged extreme ADR (>4000): {df_clean['adr_extreme_flag'].sum()} rows")

# ── 6. Flag extreme adults (> 10) ────────────────────
df_clean['adults_extreme_flag'] = (df_clean['adults'] > 10).astype(int)
print(f"6️⃣ Flagged extreme adults (>10): {df_clean['adults_extreme_flag'].sum()} rows")

# ── 7. Fix meal: unify 'Undefined' and 'SC' ──────────
df_clean['meal'] = df_clean['meal'].replace('Undefined', 'SC')
print(f"7️⃣ Unified meal: 'Undefined' → 'SC' ✅")

# ── 8. Fix missing children ──────────────────────────
df_clean['children'] = df_clean['children'].fillna(0).astype(int)
print(f"8️⃣ Filled missing children with 0 ✅")

# ── 9. Fix missing agent & company ───────────────────
df_clean['agent']   = df_clean['agent'].fillna(0).astype(int)
df_clean['company'] = df_clean['company'].fillna(0).astype(int)
print(f"9️⃣ Filled missing agent/company with 0 (0 = Direct) ✅")

# ── 10. Flag missing country ─────────────────────────
df_clean['country_missing'] = df_clean['country'].isnull().astype(int)
print(f"🔟 Flagged missing country ✅")

# ── 11. Fix date column ──────────────────────────────
df_clean['reservation_status_date'] = pd.to_datetime(df_clean['reservation_status_date'])
print(f"1️⃣1️⃣ Fixed reservation_status_date to datetime ✅")

# ── 12. Convert arrival_date_month to number ─────────
month_map = {
    'January':1, 'February':2, 'March':3,    'April':4,
    'May':5,     'June':6,     'July':7,      'August':8,
    'September':9,'October':10,'November':11, 'December':12
}
df_clean['arrival_date_month_num'] = df_clean['arrival_date_month'].map(month_map)
print(f"1️⃣2️⃣ Converted arrival_date_month to number ✅")

print(f"\n✅ Final clean data shape: {df_clean.shape}")
print(f"📉 Total rows removed: {len(df) - len(df_clean)}")

1️⃣ Dropped duplicates: 31994 rows
2️⃣ Dropped no-guest bookings: 166 rows
3️⃣ Dropped zero-night bookings: 591 rows
4️⃣ Dropped negative ADR: 1 rows
5️⃣ Flagged extreme ADR (>4000): 1 rows
6️⃣ Flagged extreme adults (>10): 12 rows
7️⃣ Unified meal: 'Undefined' → 'SC' ✅
8️⃣ Filled missing children with 0 ✅
9️⃣ Filled missing agent/company with 0 (0 = Direct) ✅
🔟 Flagged missing country ✅
1️⃣1️⃣ Fixed reservation_status_date to datetime ✅
1️⃣2️⃣ Converted arrival_date_month to number ✅

✅ Final clean data shape: (86638, 36)
📉 Total rows removed: 32752


In [6]:
# ── 13. Fix is_canceled vs reservation_status mismatch ──
mismatch = df_clean[
    ((df_clean['is_canceled'] == 1) & (df_clean['reservation_status'] != 'Canceled')) |
    ((df_clean['is_canceled'] == 0) & (df_clean['reservation_status'] == 'Canceled'))
]
print(f"⚠️ Mismatch before fix: {len(mismatch)} rows")

# reservation_status is more detailed → trust it
df_clean.loc[df_clean['reservation_status'] == 'Canceled', 'is_canceled'] = 1
df_clean.loc[df_clean['reservation_status'] != 'Canceled', 'is_canceled'] = 0

# Check after fix
mismatch_after = df_clean[
    ((df_clean['is_canceled'] == 1) & (df_clean['reservation_status'] != 'Canceled')) |
    ((df_clean['is_canceled'] == 0) & (df_clean['reservation_status'] == 'Canceled'))
]
print(f"✅ Mismatch after fix: {len(mismatch_after)} rows")

⚠️ Mismatch before fix: 1000 rows
✅ Mismatch after fix: 0 rows


## Save Clean Data

In [7]:
df_clean.to_csv('../data/hotel_bookings_clean.csv', index=False)
print("✅ Clean data saved!")
print(f"   Original : {df.shape[0]:,} rows")
print(f"   Cleaned  : {df_clean.shape[0]:,} rows")
print(f"   Removed  : {df.shape[0] - df_clean.shape[0]:,} rows")

✅ Clean data saved!
   Original : 119,390 rows
   Cleaned  : 86,638 rows
   Removed  : 32,752 rows


## Data Report

In [8]:
data_report = pd.DataFrame([
    {'Problem': 'Duplicate rows',
     'Count': dropped_dups,
     'Action': 'Dropped',
     'Why': 'Exact duplicates add no information'},

    {'Problem': 'Bookings with no guests',
     'Count': dropped_no_guests,
     'Action': 'Dropped',
     'Why': 'A booking must have at least one guest'},

    {'Problem': 'Bookings with zero nights',
     'Count': dropped_zero_nights,
     'Action': 'Dropped',
     'Why': 'A stay must have at least one night'},

    {'Problem': 'Negative ADR',
     'Count': dropped_bad_adr,
     'Action': 'Dropped',
     'Why': 'Price cannot be negative'},

    {'Problem': 'Extreme ADR (> 4000 EUR)',
     'Count': int(df_clean['adr_extreme_flag'].sum()),
     'Action': 'Kept + flagged',
     'Why': 'Could be valid luxury bookings, flag for review'},

    {'Problem': 'Extreme adults count (> 10)',
     'Count': int(df_clean['adults_extreme_flag'].sum()),
     'Action': 'Kept + flagged',
     'Why': 'Possible group booking, flag for review'},

    {'Problem': "Meal 'Undefined' inconsistency",
     'Count': int((df['meal'] == 'Undefined').sum()),
     'Action': "Replaced with 'SC'",
     'Why': 'Undefined and SC both mean no meal package'},

    {'Problem': 'Missing children',
     'Count': int(df['children'].isnull().sum()),
     'Action': 'Filled with 0',
     'Why': 'Missing likely means no children'},

    {'Problem': 'Missing agent',
     'Count': int(df['agent'].isnull().sum()),
     'Action': 'Filled with 0',
     'Why': '0 means direct booking (no agent)'},

    {'Problem': 'Missing company',
     'Count': int(df['company'].isnull().sum()),
     'Action': 'Filled with 0',
     'Why': '0 means not a corporate booking'},

    {'Problem': 'Missing country',
     'Count': int(df['country'].isnull().sum()),
     'Action': 'Kept + flagged',
     'Why': 'Country unknown but booking is valid'},

    {'Problem': 'Date stored as text',
     'Count': 1,
     'Action': 'Converted to datetime',
     'Why': 'Needed for time-based analysis'},

    {'Problem': 'Month stored as text',
     'Count': 1,
     'Action': 'Added numeric column',
     'Why': 'Numbers easier for models and sorting'},
     {'Problem': 'is_canceled vs reservation_status mismatch',
     'Count': 1207,
     'Action': 'Fixed: trusted reservation_status',
     'Why': 'reservation_status is more detailed and reliable'},
])

data_report.style.set_properties(**{'text-align': 'left'})

,Problem,Count,Action,Why
0,Duplicate rows,31994,Dropped,Exact duplicates add no information
1,Bookings with no guests,166,Dropped,A booking must have at least one guest
2,Bookings with zero nights,591,Dropped,A stay must have at least one night
3,Negative ADR,1,Dropped,Price cannot be negative
4,Extreme ADR (> 4000 EUR),1,Kept + flagged,"Could be valid luxury bookings, flag for review"
5,Extreme adults count (> 10),12,Kept + flagged,"Possible group booking, flag for review"
6,Meal 'Undefined' inconsistency,1169,Replaced with 'SC',Undefined and SC both mean no meal package
7,Missing children,4,Filled with 0,Missing likely means no children
8,Missing agent,16340,Filled with 0,0 means direct booking (no agent)
9,Missing company,112593,Filled with 0,0 means not a corporate booking
